# Run experiments on Kaggle (GPU T4)

Runs the experiments of README §7 with the `pianoabs` package. Every step is **idempotent**: outputs made with the same configuration are skipped (each output stores a config fingerprint; stale ones are recomputed), so after an interrupted session you can simply run all cells again.

Settings: *Accelerator* GPU T4, *Internet* on. Everything is written under `/kaggle/working/piano` (the repository clone), so it is kept in the notebook output.

**Part 1** (validation only) selects λ_sp and the physical learning rate for the `full` model and tunes the NMF thresholds. The chosen values are then committed to `configs/default.yaml`, and **Part 2** runs the test experiments.

In [ ]:
# 1. Code: clone the repository, or update it to the latest commit on GitHub.
#    'reset --hard' only resets tracked files (e.g. results/run_info.json written by
#    an earlier run); untracked outputs (data/, outputs/, results/thresholds) are kept.
import os, subprocess
REPO_URL = 'https://github.com/AllyPham04/piano.git'
WORK = '/kaggle/working/piano'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', REPO_URL, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', WORK, 'reset', '--hard', 'origin/main'], check=True)
os.chdir(WORK)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
# 2. System packages: FluidSynth + FluidR3_GM soundfont; check the soundfont path (README §5.5)
!apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null
!fluidsynth --version | head -1
!find / -name 'FluidR3*.sf2' 2>/dev/null

In [ ]:
# 3. Python packages (torch, librosa, soundfile are preinstalled on Kaggle)
!pip install -q -e .
!python -m pianoabs.cli info --config configs/default.yaml

In [ ]:
# 4. Data: download MAESTRO MIDI, VERIFY the frozen manifests, build GT, render C1
!python -m pianoabs.cli prepare-data --config configs/default.yaml --manifests val test --conditions C1

In [ ]:
import json, subprocess, sys

def cli(*args):
    """Run one pianoabs CLI command and stream its output; stop on error."""
    cmd = [sys.executable, '-m', 'pianoabs.cli', *args, '--config', 'configs/default.yaml']
    print('$', ' '.join(cmd[2:]))
    subprocess.run(cmd, check=True)

def cell(method, condition, variant='full', test=True):
    """Tune thresholds on validation (C3 borrows C1), then run on the test manifest."""
    if condition != 'C3':
        cli('tune', '--method', method, '--variant', variant, '--condition', condition)
    if test:
        cli('run', '--method', method, '--variant', variant, '--condition', condition,
            '--manifest', 'test')

## Part 1: selection on validation only

10 configurations of `ours` on C1 validation: λ_sp ∈ {0, 1e-3, 1e-2, 1e-1, 1} × physical learning rate ∈ {1e-2, 1e-3} (~20 min), plus NMF threshold tuning.

In [ ]:
# Part 1 was run on 2026-11-12; its result is frozen in configs/default.yaml.
# Set RUN_SELECTION = True only to reproduce the validation sweep (~25 min).
RUN_SELECTION = False
if RUN_SELECTION:
    SEL = [f'sel_lsp_{l}_lrp{r}' for r in ['1e-2', '1e-3'] for l in ['0', '1e-3', '1e-2', '1e-1', '1']]
    for v in SEL:
        cell('ours', 'C1', v, test=False)
    cell('nmf_harmonic', 'C1', test=False)

    print('\nvalidation mean onset F1 (C1, 6 excerpts)')
    for v in SEL + ['default']:
        m = 'nmf_harmonic' if v == 'default' else 'ours'
        r = json.load(open(f'results/thresholds/{m}__{v}__C1.json'))
        print(f"{m:13s} {v:22s} F1 = {r['val_mean_onset_F1']:.4f}   theta_on={r['theta_on']} "
              f"theta_rise={r['theta_rise']} reonset_ratio={r['reonset_rise_ratio']}")

Selected on validation (frozen in `configs/default.yaml`): `fit.lambda_sp = 0`, `fit.lr_physical = 1e-3`. Part 2 can now run unattended (e.g. *Save Version → Save & Run All*).

## Part 2: test experiments

In [ ]:
# E0 (C0 sanity check) and E1 (C1) for the label-free methods
cell('ours', 'C0')
cell('ours', 'C1')
cell('nmf_harmonic', 'C1')

In [ ]:
# Ablations on C1: A1 (B = 0), A2 (decay fixed), A3 (lambda_sp on test), A4 (single resolution)
for v in ['B0', 'rho_fixed', 'single_res', 'lsp_0', 'lsp_1e-3', 'lsp_1e-2', 'lsp_1e-1']:
    cell('ours', 'C1', v)

## Part 3: supervised baselines (README §4)

Default settings, no tuning. ByteDance runs in the main environment; Basic Pitch pulls in TensorFlow (numpy < 2), so it gets its own virtual environment.

In [ ]:
# ByteDance high-resolution piano transcription (downloads a 165 MB checkpoint once)
!pip install -q piano_transcription_inference
cli('run', '--method', 'bytedance_hr', '--condition', 'C1', '--manifest', 'test')

In [ ]:
# Basic Pitch in an isolated virtual environment
BP = '/kaggle/working/bpenv/bin/python'
if not os.path.exists(BP):
    subprocess.run([sys.executable, '-m', 'venv', '/kaggle/working/bpenv'], check=True)
    subprocess.run([BP, '-m', 'pip', 'install', '-q', 'basic-pitch', 'numpy<2', 'pandas<3',
                    'setuptools<81', 'pretty_midi', 'pyyaml', 'soundfile', 'tqdm'], check=True)
    subprocess.run([BP, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
subprocess.run([BP, '-c', 'import sys, numpy, basic_pitch; print(sys.version, numpy.__version__)'], check=True)
cli('run', '--method', 'basic_pitch', '--condition', 'C1', '--manifest', 'test',
    '--set', f'baselines.basic_pitch.python={BP}')

## Part 4: C3, real MAESTRO recordings (README §5.2, E2)

Streams the HF mirror `B-K/MAESTRO-test` (rows matched by MIDI md5; ~4 GB streamed, nothing kept) and crops the 30 test windows. C3 has no validation audio, so every method uses the thresholds tuned on C1.

In [ ]:
cli('fetch-real')

In [ ]:
# E2 on C3: label-free methods (thresholds from C1) and supervised baselines; A1 on C3
cell('ours', 'C3')
cell('nmf_harmonic', 'C3')
cell('ours', 'C3', 'B0')
cli('run', '--method', 'bytedance_hr', '--condition', 'C3', '--manifest', 'test')
cli('run', '--method', 'basic_pitch', '--condition', 'C3', '--manifest', 'test',
    '--set', f'baselines.basic_pitch.python={BP}')

In [ ]:
# Metrics, bootstrap CIs, Wilcoxon tests -> results/*.csv
cli('evaluate', '--manifest', 'test')

In [ ]:
# Package results for download (small files only; audio stays out)
!cd /kaggle/working && zip -qr results_bundle.zip piano/results piano/outputs/predictions piano/outputs/params && ls -la results_bundle.zip